# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 59 packages in 311ms
Checked 59 packages in 8ms


## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [2]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), and AzureOpenAI

from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

c:\Users\nawst\OneDrive\Documents\School\Fall 2026\Introduction to Agentic AI\Module 5 Agentic AI Tools\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins d9f281e8


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [4]:
persona = (
    "You are Grandmaster Victor, an elite, no-nonsense chess coach. "
    "You analyze chess games using provided PGN notation, evaluate the player's performance against core chess principles, "
    "and identify their best and worst moves with direct, honest feedback."
)

reader = PdfReader("notes/chess-notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print("Persona Loaded:")
print(persona)
print("\nNotes Preview:")
print(notes[:500], "...")

Persona Loaded:
You are Grandmaster Victor, an elite, no-nonsense chess coach. You analyze chess games using provided PGN notation, evaluate the player's performance against core chess principles, and identify their best and worst moves with direct, honest feedback.

Notes Preview:
Chess Fundamentals & Principles: A Reference Guide for Players 
1. Core Opening Principles 
• Control the Center: Occupy or influence the central squares (e4, d4, e5, d5) early 
using pawns and minor pieces. Control of the center restricts enemy mobility and 
creates space for your own pieces. 
• Rapid Piece Development: Bring your knights and bishops out early. As a rule of 
thumb, develop knights before bishops so you can accurately determine the best 
diagonal for your bishops. Avoid moving t ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [5]:
system_prompt = f"""
# Your role

You are {persona}

You review chess games using the reference notes below.

# Context

Here are the chess fundamentals, principles, and evaluation criteria you are using to evaluate the game:

{notes}

# Rules

Analyze the user's game or move sequence based only on the chess principles above. Keep your feedback concise, professional, and firmly in character as a Grandmaster.

After the user provides a PGN or describes a move, evaluate their performance. Identify their **best move** and their **worst move/mistake** based on the principles, and give a clear explanation for both.

IMPORTANT: If the user makes a significant mistake, blunder, or inaccuracy, you must record it using your custom logging tool (`save_chess_analysis`) before concluding your response or moving on.

Always stay in character as Grandmaster Viktor. If the user asks about topics completely unrelated to chess or the provided notes, state that it is outside your coaching scope.
"""

display(Markdown(system_prompt))


# Your role

You are You are Grandmaster Victor, an elite, no-nonsense chess coach. You analyze chess games using provided PGN notation, evaluate the player's performance against core chess principles, and identify their best and worst moves with direct, honest feedback.

You review chess games using the reference notes below.

# Context

Here are the chess fundamentals, principles, and evaluation criteria you are using to evaluate the game:

Chess Fundamentals & Principles: A Reference Guide for Players 
1. Core Opening Principles 
• Control the Center: Occupy or influence the central squares (e4, d4, e5, d5) early 
using pawns and minor pieces. Control of the center restricts enemy mobility and 
creates space for your own pieces. 
• Rapid Piece Development: Bring your knights and bishops out early. As a rule of 
thumb, develop knights before bishops so you can accurately determine the best 
diagonal for your bishops. Avoid moving the same piece twice in the opening if 
possible. 
• King Safety (Castle Early): Get your king to safety and activate your rooks by 
castling within the first 10 moves. Leaving your king in the center makes it vulnerable 
to tactical strikes and open-file attacks. 
• Avoid Premature Queen Moves: Bringing your queen out too early allows your 
opponent to attack it with minor pieces, gaining free tempo (time) while forcing your 
queen back. 
2. Tactical Motifs and Common Blunders 
• The Fork: A single piece (often a knight or pawn) attacks two or more enemy pieces 
simultaneously. Knights are notorious for devastating forks (especially "royal forks" 
hitting the king and queen). 
• The Pin: An attacking piece threatens an enemy piece that cannot move without 
exposing a more valuable piece behind it (typically the king or queen). Absolute pins 
involve the king; relative pins involve other pieces. 
• Discovered Attacks / Checks: Moving one piece out of the way to instantly reveal 
an attack or check from a piece behind it (like a rook, bishop, or queen). 
• Hanging Pieces: Leaving a piece undefended or insufficiently protected, allowing 
the opponent to capture it for free or for a favorable material trade. 
• Overloaded Defenders: A defensive piece tasked with protecting too many squares 
or pieces at once, causing a breakdown when pressure is applied. 
3. Basic Evaluation Criteria for Moves 
• Best Move: A move that strictly adheres to positional principles, wins material, 
improves piece activity, or creates an unstoppable tactical threat. • Inaccuracy: A move that is sub-optimal and slightly worsens your position or gives 
your opponent a minor advantage. 
• Mistake / Blunder: A severe error that loses material, blows a winning advantage, 
walks into a tactical trap, or hands the opponent a decisive win. 
 

# Rules

Analyze the user's game or move sequence based only on the chess principles above. Keep your feedback concise, professional, and firmly in character as a Grandmaster.

After the user provides a PGN or describes a move, evaluate their performance. Identify their **best move** and their **worst move/mistake** based on the principles, and give a clear explanation for both.

IMPORTANT: If the user makes a significant mistake, blunder, or inaccuracy, you must record it using your custom logging tool (`save_chess_analysis`) before concluding your response or moving on.

Always stay in character as Grandmaster Viktor. If the user asks about topics completely unrelated to chess or the provided notes, state that it is outside your coaching scope.


## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the review"},
]

response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [6]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content

In [7]:
chat("I'm ready, let's start the review", [])

'Good. Send me the PGN or the full move list.\n\nI’ll review it like a professional:\n- opening discipline\n- tactical mistakes\n- best move\n- worst move\n- direct verdict on your play\n\nIf there’s a serious blunder, I’ll call it exactly what it is.'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [8]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [19]:
def save_chess_analysis(move_summary, game_label="Game #1", lichess_url="https://lichess.org/"):
    """Logs a chess analysis along with a game label and a direct Lichess link."""
    print(f"Tool called to record chess analysis for {game_label}")
    
    formatted_entry = (
        f"--- {game_label} ---\n"
        f"Analysis: {move_summary}\n"
        f"Lichess Link: {lichess_url}\n"
        f"\n"  # This creates the clean blank line at the very end
    )
    
    with open("chess_review_log.txt", "a", encoding="utf-8") as f:
        f.write(formatted_entry)
        
    return "Chess analysis logged successfully with title and link"

In [10]:
save_chess_analysis("test value")  # quick manual test of your tool

Tool called to record chess error: test value


'Chess mistake logged for review'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [20]:
my_tool_json = {
    "name": "save_chess_analysis",
    "description": "Use this tool to record significant chess mistakes, blunders, or inaccuracies found during a game review. Provide the analysis summary, a game label (like Game #1), and a Lichess link if available.",
    "parameters": {
        "type": "object",
        "properties": {
            "move_summary": {
                "type": "string", 
                "description": "A description of the mistake, blunder, or bad move along with the correct principle."
            },
            "game_label": {
                "type": "string",
                "description": "A label or title for the game, e.g., 'Game #1' or 'Sicilian Defense Practice'."
            },
            "lichess_url": {
                "type": "string",
                "description": "The URL to the game on Lichess."
            }
        },
        "required": ["move_summary"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": my_tool_json}]

`## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.`

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        
        # Parse the arguments matching what you defined in your tool schema
        move_summary = json.loads(tool_call.function.arguments).get("move_summary")
        
        # Call your custom chess logging tool
        save_chess_analysis(move_summary)
        
        # Append the tool call and tool result back into the message history
        messages.append(message_obj)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Chess analysis logged for review"})
        
        # Call the model again so it can turn that result into a normal reply in character
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

In [12]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\nawst\OneDrive\Documents\School\Fall 2026\Introduction to Agentic AI\Module 5 Agentic AI Tools\.venv\Lib\site-packages\gradio\queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\nawst\OneDrive\Documents\School\Fall 2026\Introduction to Agentic AI\Module 5 Agentic AI Tools\.venv\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\nawst\OneDrive\Documents\School\Fall 2026\Introduction to Agentic AI\Module 5 Agentic AI Tools\.venv\Lib\site-packages\gradio\blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\nawst\OneDrive\Do

## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [21]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        
        for tool_call in message_obj.tool_calls:
            args = json.loads(tool_call.function.arguments)
            move_summary = args.get("move_summary")
            game_label = args.get("game_label", "Game #1")
            lichess_url = args.get("lichess_url", "https://lichess.org/")
            
            # Execute your custom tool with all fields
            save_chess_analysis(move_summary, game_label, lichess_url)
            
            messages.append({
                "role": "tool", 
                "tool_call_id": tool_call.id, 
                "content": "Chess analysis logged for review"
            })
            
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tool called to record chess analysis for Game ECw8MEqT
Tool called to record chess analysis for Game lI1yY8TT


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make
   sense for this topic?
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What,
   if anything, did you have to think about differently?
3. What would break if you asked your Persona a question completely unrelated to your notes? Did
   your system prompt handle that the way you wanted?
4. If you added a second tool to this agent, what would it be, and what would change in Steps
   9–11 to support it?

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.